<a href="https://colab.research.google.com/github/ewxxd/SIT215/blob/main/222461154.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import heapq
import math
from typing import Dict, List, Tuple, Optional, Set

Edge = Tuple[str, str, int]

class Environment:

    def __init__(self):

      self.NODE_COORDS = {
            "N1": (0, 5),
            "N2": (1, 5),
            "N3": (2, 5),
            "N4": (0, 4),
            "N5": (0, 3),
            "N6": (1, 4),
            "N7": (2, 4),
            "N8": (3, 4),
            "N9": (3, 3),
            "N10": (4, 3),
            "N11": (2, 3),
            "N12": (4, 4),
            "N13": (5, 3),
            "N14": (5, 2),
            "N15": (6, 3),
            "N16": (4, 1),
            "N17": (5, 1),
            "N18": (6, 1),
            "N19": (6, 2),
            "N20": (7, 1)
        }


      self.EDGES: List[Edge] = [
            ("N1", "N2", 100),
            ("N1", "N4", 200),
            ("N1", "N6", 180),
            ("N2", "N3", 110),
            ("N3", "N8", 110),
            ("N4", "N6", 180),
            ("N4", "N7", 250),
            ("N4", "N8", 300),
            ("N4", "N9", 220),
            ("N4", "N10", 150),
            ("N5", "N7", 50),
            ("N7", "N8", 350),
            ("N7", "N9", 270),
            ("N7", "N10", 200),
            ("N7", "N11", 100),
            ("N8", "N9", 120),
            ("N8", "N10", 250),
            ("N8", "N11", 350),
            ("N8", "N12", 100),
            ("N9", "N10", 170),
            ("N10", "N15", 140),
            ("N11", "N19", 250),
            ("N12", "N14", 100),
            ("N13", "N14", 100),
            ("N13", "N18", 150),
            ("N14", "N17", 150),
            ("N15", "N19", 180),
            ("N15", "N20", 150),
            ("N16", "N17", 100),
            ("N17", "N18", 100),
            ("N18", "N20", 80),
            ("N19", "N20", 170)
        ]

        # Heuristic distance to goal N20
      self.N1_TO_N20 = {
            "N1": 1000,
            "N2": 900,
            "N3": 800,
            "N4": 700,
            "N5": 600,
            "N6": 700,
            "N7": 500,
            "N8": 400,
            "N9": 300,
            "N10": 200,
            "N11": 400,
            "N12": 300,
            "N13": 200,
            "N14": 200,
            "N15": 150,
            "N16": 300,
            "N17": 200,
            "N18": 100,
            "N19": 150,
            "N20": 0
        }

        # Static constraints
      self.staticConstraints: List[Tuple[str, str]] = [
            ("N8", "N12"),
            ("N10", "N15"),
            ("N13", "N18")
        ]

        # Dynamic constraints
      self.dynamicConstraints: List[Tuple[str, str]] = [
            ("N1", "N4"),
            ("N4", "N6"),
            ("N4", "N8"),
            ("N7", "N8"),
            ("N8", "N9"),
            ("N8", "N11"),
            ("N15", "N19"),
            ("N19", "N20")
        ]


def usingDynamic(a: str, b: str, env: Environment, dynamic=False):
    if (a, b) in env.staticConstraints or (b, a) in env.staticConstraints:
      return True

    if dynamic:
      if (a, b) in env.dynamicConstraints or (b, a) in env.dynamicConstraints:
        return True
    return False



def build_undirected_graph(edges: List[Edge]) -> Dict[str, Dict[str, int]]:
    graph: Dict[str, Dict[str, int]] = {}
    for a, b, w in edges:
        graph.setdefault(a, {})[b] = w
        graph.setdefault(b, {})[a] = w
    return graph



def reconstruct_path(parents: Dict[str, Optional[str]], goal: str) -> List[str]:
    path = []
    cur: Optional[str] = goal
    while cur is not None:
        path.append(cur)
        cur = parents.get(cur)
    return list(reversed(path))


def manhattan(a: str, b: str, env: Environment):
    x1, y1 = env.NODE_COORDS[a]
    x2, y2 = env.NODE_COORDS[b]
    return (abs(x1 - x2) + abs(y1 - y2)) * 100


def euclidean(a: str, b: str, env: Environment):
    #Euclidean distance
    x1, y1 = env.NODE_COORDS[a]
    x2, y2 = env.NODE_COORDS[b]

    dx = x2 - x1
    dy = y2 - y1

    return math.sqrt(dx*dx + dy*dy)


def ucs(start: str, goal: str, graph: Dict[str, Dict[str, int]], env: Environment, use_dynamic=False):
    frontier = []
    heapq.heappush(frontier, (0, start))
    parents = {start: None}
    best_g = {start: 0}
    expanded = []
    explored: Set[str] = set()

    while frontier:
        g, node = heapq.heappop(frontier)
        if node in explored:
            continue
        explored.add(node)
        expanded.append(node)

        if node == goal:
            return reconstruct_path(parents, goal), g, expanded

        for nb, w in graph[node].items():
            if usingDynamic(node, nb, env, dynamic=use_dynamic):
              continue

            ng = g + w
            if nb not in best_g or ng < best_g[nb]:
                best_g[nb] = ng
                parents[nb] = node
                heapq.heappush(frontier, (ng, nb))
    return None, None, expanded


def astar(start: str, goal: str, graph: Dict[str, Dict[str, int]], env: Environment,
          heuristic_type="manhattan", use_dynamic=False):

    frontier = []
    parents = {start: None}
    best_g = {start: 0}
    expanded = []
    explored = set()

    h = lambda n: manhattan(n, goal, env) if heuristic_type == "manhattan" else euclidean(n, goal, env) if heuristic_type == "euclidean" else 0

    heapq.heappush(frontier, (h(start), 0, start))

    while frontier:
        f, g, node = heapq.heappop(frontier)

        if node in explored:
            continue

        explored.add(node)
        expanded.append(node)

        if node == goal:
            return reconstruct_path(parents, goal), g, expanded

        for nb, w in graph[node].items():

            if usingDynamic(node, nb, env, dynamic=use_dynamic):
                continue

            ng = g + w
            nf = ng + h(nb)

            if nb not in best_g or ng < best_g[nb]:
                best_g[nb] = ng
                parents[nb] = node
                heapq.heappush(frontier, (nf, ng, nb))

    return None, None, expanded



env = Environment()
graph = build_undirected_graph(env.EDGES)

tests = [
    ("N1", "N20"),
    ("N5", "N19"),
    ("N2", "N18")
]

for start, goal in tests:

    #enviroment is relaxed, dynamic constraints are turned off
    path_u, cost_u, exp_u = ucs(start, goal, graph, env, use_dynamic=False)
    path_a1, cost_a1, exp_a1 = astar(start, goal, graph, env, "manhattan", use_dynamic=False)

    print(f"\n {start} to {goal} ")

    print("UCS:")
    print(" Path:", path_u)
    print(" Cost:", cost_u)
    print(" Expanded:", len(exp_u))

    print("\nA*:")
    print(" Path:", path_a1)
    print(" Cost:", cost_a1)
    print(" Expanded:", len(exp_a1))

# HEURISTIC COMPARISON
print("\nHeuristic Comparison (N1 to N20)")

for mode in ["manhattan", "euclidean"]:
    path, cost, expanded = astar("N1", "N20", graph, env, mode, use_dynamic=False)

    print("\nMode:", mode)
    print("Path:", path)
    print("Cost:", cost)
    print("Expanded:", len(expanded))


print("\nHeuristic Comparison (N2 to N17)")

for mode in ["manhattan", "euclidean"]:
    path, cost, expanded = astar("N2", "N17", graph, env, mode, use_dynamic=False)

    print("\nMode:", mode)
    print("Path:", path)
    print("Cost:", cost)
    print("Expanded:", len(expanded))









 N1 to N20 
UCS:
 Path: ['N1', 'N4', 'N7', 'N11', 'N19', 'N20']
 Cost: 970
 Expanded: 13

A*:
 Path: ['N1', 'N4', 'N7', 'N11', 'N19', 'N20']
 Cost: 970
 Expanded: 12

 N5 to N19 
UCS:
 Path: ['N5', 'N7', 'N11', 'N19']
 Cost: 400
 Expanded: 7

A*:
 Path: ['N5', 'N7', 'N11', 'N19']
 Cost: 400
 Expanded: 5

 N2 to N18 
UCS:
 Path: ['N2', 'N3', 'N8', 'N11', 'N19', 'N20', 'N18']
 Cost: 1070
 Expanded: 15

A*:
 Path: ['N2', 'N3', 'N8', 'N11', 'N19', 'N20', 'N18']
 Cost: 1070
 Expanded: 11

Heuristic Comparison (N1 to N20)

Mode: manhattan
Path: ['N1', 'N4', 'N7', 'N11', 'N19', 'N20']
Cost: 970
Expanded: 12

Mode: euclidean
Path: ['N1', 'N4', 'N7', 'N11', 'N19', 'N20']
Cost: 970
Expanded: 13

Heuristic Comparison (N2 to N17)

Mode: manhattan
Path: ['N2', 'N3', 'N8', 'N11', 'N19', 'N20', 'N18', 'N17']
Cost: 1170
Expanded: 14

Mode: euclidean
Path: ['N2', 'N3', 'N8', 'N11', 'N19', 'N20', 'N18', 'N17']
Cost: 1170
Expanded: 16
